# GPU run on Colab (T4)

Runtime → Change runtime type → **T4 GPU**. The package is the artefact; this notebook only drives the CLI. Total run time is well under an hour.

*Research project, not a medical device.*

In [ ]:
!nvidia-smi -L
!git clone https://github.com/marselaneziraj/pneumonia-detection-xray.git
%cd pneumonia-detection-xray
!pip install -q -e ".[data]"

## Kaggle token
Upload `kaggle.json` (kaggle.com → Settings → API → Create New Token).

In [ ]:
from google.colab import files
import os, shutil
up = files.upload()  # choose kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data --unzip
!kaggle datasets metadata paultimothymooney/chest-xray-pneumonia -p data && cat data/dataset-metadata.json | head -40
!date -u +%F  # record as the retrieval date in README

In [ ]:
!pneumonia prepare --config configs/densenet121.yaml

In [ ]:
!pneumonia train --config configs/densenet121.yaml

In [ ]:
!pneumonia train --config configs/resnet18.yaml

In [ ]:
!pneumonia train --config configs/resnet18_scratch.yaml

In [ ]:
!pneumonia evaluate
!pneumonia gradcam --per-case 3
!pneumonia report

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/gradcam_panel.png', width=900))
display(Image('results/figures/threshold_operating_curve.png', width=700))

## Take the results home
Download the zip and unpack it over your local clone. It contains the results, the figures, the split manifests and the docs with their filled-in tables, but no weights.

In [ ]:
!zip -qr results_bundle.zip results splits README.md report.md docs/model_card.md runs/*/history.csv runs/*/train_meta.json -x 'results/predictions/*'
files.download('results_bundle.zip')